# A17 · 广告实验、偏差与反作弊

先修：公用31–34；A16。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

曝光选择会改变人群混合，即使各层表现相同，总体CTR也能出现差异。请求级随机桶共享活动预算时会互相影响，需要活动或市场级设计并考虑干扰。反作弊阈值是调查线索，不是事实标签；异常流量剔除策略应预先定义，同时报告误伤与业务护栏。特征哈希切分只阻止相同向量跨区，不能证明用户独立或公平。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
a=ac.experiment_bias();print(a)
assert a['aggregate']['A']!=a['aggregate']['B']
assert np.isclose(a['standardized']['A'],a['standardized']['B'])
assert a['false_positive']==1
print('筛查precision',a['true_positive']/(a['true_positive']+a['false_positive']))
print('共享预算先到先得',ac.shared_budget_interference(),ac.shared_budget_interference(('B','B','A','A')))
assert ac.shared_budget_interference()['B']==0

{'aggregate': {'A': 0.19, 'B': 0.11}, 'standardized': {'A': 0.15000000000000002, 'B': 0.15000000000000002}, 'flagged': ['bot', 'campaign_burst'], 'true_positive': 1, 'false_positive': 1, 'interference': 'Shared campaign budgets couple request buckets; use campaign/market randomization or model spillovers.'}
筛查precision 0.5
共享预算先到先得 {'A': 2, 'B': 0} {'A': 0, 'B': 2}


## 解释与局限

统一分层权重后差异消失；这不是证明任意真实效果都由混合造成，而是一个可反驳简单总体比较的反例。高频规则误伤正常活动突发，不能在未知标签上宣称达到真实反作弊准确率。

## 查看源码，再做练习

设计实验的主指标、预算和用户体验护栏、随机化单位和停止规则；模拟共享预算耗尽如何污染请求桶。

In [3]:
import inspect
print(inspect.getsource(ac.experiment_bias))

def experiment_bias():
    # Group A/B have identical stratum rates; unequal mix creates aggregate difference.
    counts={'A':[(90,18),(10,1)],'B':[(10,2),(90,9)]}
    aggregate={k:sum(y for n,y in cells)/sum(n for n,y in cells) for k,cells in counts.items()}
    standardized={k:sum(y/n for n,y in cells)/2 for k,cells in counts.items()}
    # A rule is only a triage signal; legitimate burst is a false positive.
    traffic=[('bot',50,True),('human',1,False),('campaign_burst',40,False)]
    flags=[name for name,rate,_ in traffic if rate>20]
    return dict(aggregate=aggregate,standardized=standardized,flagged=flags,true_positive=1,false_positive=1,
                interference='Shared campaign budgets couple request buckets; use campaign/market randomization or model spillovers.')



## 来源与下一步

A-LIU其他技术目录；分层反例、实验干扰与误伤是Lab自编。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。